In [ ]:
# Linalg
import numpy as np 
import torch
from torch import nn

# Data handling
import h5py as hf
from torch.utils.data import DataLoader, Dataset

# Plotting
import matplotlib.pyplot as plt
from rich.progress import track
import seaborn as sns

In [ ]:
# set the device
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using {device} device")

# Kernel Construction

In [ ]:
def normalize_positions(positions, kernel_centers):
    """
    Normalize positions and kernel centers to the range [0, 1].

    Parameters:
    positions (numpy.ndarray): Array of shape (N, D) for particle positions.
    kernel_centers (numpy.ndarray): Array of shape (M, D) for kernel centers.

    Returns:
    tuple: (normalized_positions, normalized_kernel_centers, scale_factors)
    """
    mins = np.min(positions, axis=0)
    maxs = np.max(positions, axis=0)
    scale_factors = maxs - mins

    normalized_positions = (positions - mins) / scale_factors
    normalized_kernel_centers = (kernel_centers - mins) / scale_factors

    return normalized_positions, normalized_kernel_centers, scale_factors

In [ ]:
def gaussian_kernel_vector_normalized(positions, velocities, kernel_centers, kernel_widths):
    """
    Compute Gaussian kernel vector after normalizing positions and kernel centers.

    Parameters:
    positions (numpy.ndarray): Array of shape (N, 2) for particle positions.
    velocities (numpy.ndarray): Array of shape (N, 2) for particle velocities.
    kernel_centers (numpy.ndarray): Array of shape (M, 2) for kernel centers.
    kernel_widths (numpy.ndarray): Array of shape (M,) for kernel widths.

    Returns:
    numpy.ndarray: Concatenated vector of kernel responses (r1, r2, r3).
    """
    # Normalize positions and kernel centers
    normalized_positions, normalized_kernel_centers, scale_factors = normalize_positions(positions, kernel_centers)
    normalized_widths = kernel_widths / np.mean(scale_factors)

    # Compute kernels in normalized space
    kernels = np.exp(-np.linalg.norm(normalized_positions[:, None, :] - normalized_kernel_centers[None, :, :], axis=-1)**2 /
                     (2 * normalized_widths**2))

    # Compute r1, r2, r3
    r1 = np.sum(kernels, axis=0)
    r2 = np.sum(kernels * velocities[:, 0][:, None], axis=0)
    r3 = np.sum(kernels * velocities[:, 1][:, None], axis=0)

    return np.concatenate([r1, r2, r3])

In [ ]:
def compute_velocities(positions, dt):
    """
    Compute velocities from positions over time.

    Parameters:
    positions (numpy.ndarray): Array of shape (T, N, D) containing the positions of N particles in D dimensions over T time steps.
    dt (float): Time step between frames.

    Returns:
    numpy.ndarray: Array of shape (T, N, D) containing the velocities, where the first and last time steps are filled using forward and backward differences respectively.
    """
    # Compute central differences for velocities
    velocities = np.zeros_like(positions)
    velocities[1:-1] = (positions[2:] - positions[:-2]) / (2 * dt)

    # Forward difference for the first frame
    velocities[0] = (positions[1] - positions[0]) / dt

    # Backward difference for the last frame
    velocities[-1] = (positions[-1] - positions[-2]) / dt

    return velocities


In [ ]:
with hf.File("/work/stovey/konstanz-reservoir/first_tries/strong_coupling_modulation_ind_0.5/trajectories.h5", "r") as db:
    positions = np.concatenate([db["particles_x"][:][:, :, None], db["particles_y"][:][:, :, None]], axis=-1)
    signal = db["original_signal"][:]

In [ ]:
where_are_NaNs = np.isnan(positions)
positions[where_are_NaNs] = 0

In [ ]:
velocities = compute_velocities(positions, 1.0)

In [ ]:
indices = np.random.randint(0, positions.shape[1] - 1, 50)

In [ ]:
domain_size = (np.max(positions[:, 0]) - np.min(positions[:, 0])) * (np.max(positions[:, 1]) - np.min(positions[:, 1]))
resolution = 1
width = np.sqrt(domain_size) / resolution

In [ ]:
kernels = []
for p, v in zip(positions, velocities):
    kernels.append(gaussian_kernel_vector_normalized(p, v, np.take(p, indices, axis=0), width))

kernels = np.array(kernels)

In [ ]:
sns.histplot(kernels[0], stat="density")
sns.histplot(np.concatenate(kernels), stat="density")
plt.xscale("log")

# Model Training

In [ ]:
class NeuralNetwork(nn.Module):
    
    def __init__(self, state_dimension: int, output_dimension: int):
        """
        Build the network.
        
        Parameters
        ----------
        state_dimension : int
                Dimension of the state representation.
                This is the input to the layer.
        output_dimension : int
                Dimension of the output being predicted.
        """
        super().__init__()

        self.readout_layer = nn.Sequential(
            nn.Linear(state_dimension, output_dimension),
        )

    def forward(self, x):
        """
        Forward pass through the network.
        
        As we are doing reservoir computing, this is 
        simply a linear layer.
        """
        return self.readout_layer(x)

In [ ]:
class ReservoirDataset(Dataset):
    """
    Custom dataset for the training.
    """
    def __init__(
        self, 
        state_data: np.ndarray, 
        prediction_length: int,
        function_data: np.ndarray,
    ):
        """
        Constructor for the dataset.

        Parameters
        ----------
        state_data : np.ndarray
                State description data.
        function_data : np.ndarray
                Function data being fit.
                This will be the target data.
        prediction_length : int
                How far into the future you will predict.
        """
        self.state_data = torch.Tensor(state_data[:-prediction_length]).to(device)

        self.function_data = torch.Tensor(
            function_data[prediction_length:]
        ).to(device)

    def __len__(self):
        """
        Length of the dataset.
        """
        return int(
            len(self.function_data)
        )
    
    def __getitem__(self, idx: int):
        """
        Collect an item from the dataset.
        
        Parameters
        ----------
        idx : int
                Index of the state to take.
        """
        state = self.state_data[idx]
        target = self.function_data[idx]
        
        return state, target

In [ ]:
positions[-10:]

In [ ]:
def train(dataloader, model, loss_fn, optimizer):
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)
        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss = loss.item()
            
    return loss

def test(dataloader, model, loss_fn) -> np.ndarray:
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss = 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
    test_loss /= num_batches
    
    return test_loss

In [ ]:
def train_model(dataset, test_ds, model = None):
    """
    Train a model on the current data.
    """    
    if model is None:
        model = NeuralNetwork(
            state_dimension=150,
            output_dimension=1
        ).to(device)

        model = model.type(torch.float32)

    # Use MSE loss
    loss_fn = nn.MSELoss()

    # Use ADAM optimizer
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)

    # Create the loader
    loader = DataLoader(dataset, batch_size=64, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=250, shuffle=False)
    
    # Train the network
    epochs = 2000
    train_losses = []
    test_losses = []

    for t in track(range(epochs)):
        loss = train(loader, model, loss_fn, optimizer)
        train_losses.append(loss)
        loss = test(test_loader, model, loss_fn)
        test_losses.append(loss)

    train_losses = [item.cpu().detach().numpy() for item in train_losses]
    
    return train_losses, test_losses, model


In [ ]:
prediction_length = 2
split_offset=6000

train_ds = ReservoirDataset(
    state_data=kernels[:split_offset, :],
    function_data=signal[:split_offset],
    prediction_length=prediction_length
)

test_ds = ReservoirDataset(
    state_data=kernels[split_offset:, :],
    function_data=signal[split_offset:],
    prediction_length=prediction_length
)

train_losses, test_losses, model = train_model(train_ds, test_ds, model=None)


In [ ]:
plt.plot(train_losses, label="Train")
plt.plot(test_losses, label="Test")
plt.yscale("log")
plt.legend()
plt.show()

In [ ]:
prediction = model(torch.tensor(kernels[:100]).to(device).type(torch.float32))

In [ ]:
plt.plot(signal[:100])
plt.plot(prediction.cpu().detach().numpy())